# RS ai — آزمایش موتور آواتار روی GPU ابری رایگان

**چرا این دفترچه؟** رایانهٔ محلی شما فقط Intel HD Graphics 4000 دارد و برای اجرای OmniAvatar با CUDA مناسب نیست. این دفترچه تلاش می‌کند مرحلهٔ تولید آواتار را روی GPU ابری اجرا کند و ویدئوی خروجی را برای دانلود برگرداند.

**محدودیت مهم:** دسترسی GPU رایگان Colab تضمین‌شده نیست. این دفترچه با تنظیمات سبک برای آزمایش اولیه نوشته شده و اجرای موفق OmniAvatar روی GPU رایگان هنوز تضمین نشده است. ابتدا فقط یک صدای ۵ ثانیه‌ای آزمایش کنید؛ بعد از موفقیت، محدودیت زمانی را بردارید.

**حریم خصوصی:** عکس و صدا را به محیط ابری منتقل می‌کنید. فقط فایلی را آپلود کنید که با پردازش آن در سرویس ابری موافقید.

## روش استفاده
1. این فایل را در Google Colab باز کنید و از `Runtime → Change runtime type` گزینهٔ GPU را انتخاب کنید.
2. سلول‌ها را به ترتیب اجرا کنید. دانلود مدل‌ها در محیط ابری انجام می‌شود، نه روی رایانهٔ شما.
3. در سلول ورودی، عکس و فایل صوتی را انتخاب کنید.
4. اگر تولید آزمایشی موفق شد، در سلول ورودی مقدار `TEST_SECONDS = None` بگذارید تا کل فایل صوتی پردازش شود؛ این حالت می‌تواند به‌مراتب طولانی‌تر باشد و سهمیهٔ رایگان را مصرف کند.

مدل مرجع و کد رسمی: https://github.com/Omni-Avatar/OmniAvatar

In [ ]:
# سلول ۱: بررسی اینکه واقعاً GPU به محیط ابری اختصاص داده شده باشد
import subprocess, sys
r = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
print(r.stdout if r.returncode == 0 else r.stderr)
if r.returncode != 0:
    raise RuntimeError('GPU در این نشست در دسترس نیست. Runtime را روی GPU بگذارید و دوباره اجرا کنید؛ مدل‌ها را هنوز دانلود نکنید.')

In [ ]:
# سلول ۲: دریافت سورس و وابستگی‌های رسمی OmniAvatar
# دانلودها در فضای موقت Colab انجام می‌شوند، نه روی رایانهٔ ویندوزی شما.
import os, pathlib, subprocess, textwrap
WORK = pathlib.Path('/content/RS-ai-cloud')
REPO = pathlib.Path('/content/OmniAvatar')
WORK.mkdir(parents=True, exist_ok=True)
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/Omni-Avatar/OmniAvatar',str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','--upgrade','pip','setuptools','wheel'], check=True)
# Official repo pins this CUDA-enabled PyTorch family. If Colab's runtime is incompatible,
# stop here and keep the full error; do not repeatedly reinstall packages blindly.
subprocess.run([sys.executable,'-m','pip','install','torch==2.4.0','torchvision==0.19.0','torchaudio==2.4.0','--index-url','https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt','huggingface_hub[cli]'], check=True)
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch به GPU دسترسی ندارد؛ قبل از دانلود مدل‌ها مشکل محیط را برطرف کنید.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# سلول ۳: دانلود وزن‌های موردنیاز؛ ممکن است چندین گیگابایت باشد
# این مرحله را فقط در محیط ابری دارای GPU اجرا کنید.
env = os.environ.copy()
env['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
model_dir = REPO / 'pretrained_models'
model_dir.mkdir(exist_ok=True)
downloads = [
    ['hf','download','Wan-AI/Wan2.1-T2V-1.3B','--local-dir',str(model_dir/'Wan2.1-T2V-1.3B')],
    ['hf','download','facebook/wav2vec2-base-960h','--local-dir',str(model_dir/'wav2vec2-base-960h')],
    ['hf','download','OmniAvatar/OmniAvatar-1.3B','--local-dir',str(model_dir/'OmniAvatar-1.3B')],
]
for cmd in downloads:
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
print('\nDownload step completed. Disk usage:')
subprocess.run(['du','-sh',str(model_dir)], check=False)

In [ ]:
# سلول ۴: انتخاب عکس و صدا و ساخت ورودی موتور
from google.colab import files
from IPython.display import Video, display
import shutil, re
uploaded = files.upload()
image_exts = {'.png','.jpg','.jpeg','.webp'}
audio_exts = {'.wav','.mp3','.m4a','.aac','.flac','.ogg'}
image_files = [pathlib.Path('/content')/n for n in uploaded if pathlib.Path(n).suffix.lower() in image_exts]
audio_files = [pathlib.Path('/content')/n for n in uploaded if pathlib.Path(n).suffix.lower() in audio_exts]
if not image_files or not audio_files:
    raise ValueError('یک عکس JPG/PNG و یک فایل صوتی WAV/MP3 را با همین سلول بارگذاری کنید.')
image_in, audio_in = image_files[0], audio_files[0]
job = WORK / 'input'
job.mkdir(parents=True, exist_ok=True)
image_path = job / ('reference' + image_in.suffix.lower())
shutil.copy2(image_in, image_path)
audio_path = job / 'speech_16k.wav'
TEST_SECONDS = 5  # پس از موفقیت آزمایشی، None بگذارید تا کل صدا پردازش شود.
ffmpeg_cmd = ['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(audio_in),'-vn','-ac','1','-ar','16000']
if TEST_SECONDS is not None:
    ffmpeg_cmd += ['-t', str(TEST_SECONDS)]
ffmpeg_cmd += [str(audio_path)]
subprocess.run(ffmpeg_cmd, check=True)
PROMPT = ('Photorealistic adult presenter speaking naturally to camera; preserve the identity and appearance of the reference person; natural head movement and expressive but plausible hand and arm gestures; stable camera; coherent body motion; no text, subtitles or logos.')
PROMPT = PROMPT.replace('@@',' ')
input_file = WORK / 'rsai_input.txt'
input_file.write_text(f'{PROMPT}@@{image_path}@@{audio_path}\n', encoding='utf-8')
print('Reference image:', image_path.name)
print('Audio duration mode:', f'{TEST_SECONDS} seconds test' if TEST_SECONDS else 'full audio')
print('Input file prepared:', input_file)

In [ ]:
# سلول ۵: اجرای آزمایشی مدل. تنظیمات کم‌حافظه و تعداد گام پایین برای اولین تست.
# این تنظیمات ممکن است کیفیت پایین‌تری بدهند؛ برای کیفیت بیشتر NUM_STEPS را به 20 یا بالاتر افزایش دهید.
NUM_STEPS = 8
MAX_TOKENS = 15000
HYPARAMS = f'dtype=fp16,num_steps={NUM_STEPS},max_tokens={MAX_TOKENS},num_persistent_param_in_dit=0,tea_cache_l1_thresh=0.14'
cmd = ['torchrun','--standalone','--nproc_per_node=1','scripts/inference.py','--config','configs/inference_1.3B.yaml','--input_file',str(input_file),'--hparams='+HYPARAMS]
print('Running inference. If it reports CUDA out-of-memory, this free GPU is not sufficient for this configuration.')
print('Command:', ' '.join(cmd))
run = subprocess.run(cmd, cwd=str(REPO), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_path = WORK / 'omniavatar.log'
log_path.write_text(run.stdout or '', encoding='utf-8', errors='replace')
print((run.stdout or '')[-12000:])
if run.returncode != 0:
    raise RuntimeError(f'OmniAvatar failed with code {run.returncode}. Full log: {log_path}')
outputs = sorted((REPO/'demo_out').rglob('result_000.mp4'), key=lambda p:p.stat().st_mtime, reverse=True)
if not outputs:
    raise FileNotFoundError('Inference returned successfully but result_000.mp4 was not found. Inspect the log.')
video_path = outputs[0]
print('Generated video:', video_path, f'({video_path.stat().st_size:,} bytes)')
display(Video(str(video_path), embed=True))

In [ ]:
# سلول ۶: دانلود MP4 به رایانه
download_path = WORK / 'RS-ai-avatar-cloud.mp4'
shutil.copy2(video_path, download_path)
files.download(str(download_path))